[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Zgraph/blob/main/zgraph/examples/binary.ipynb)

In [1]:
import sys

# Check if Zgraph is already installed
try:
    import zgraph
    print("Zgraph is already installed.")
except ImportError:
    if 'google.colab' in sys.modules:
        print("Installing Zgraph...")
        !pip install -q "git+https://github.com/themintlab/Zgraph.git#subdirectory=zgraph"
        print("Successfully installed Zgraph!")

Zgraph is already installed.


In [2]:
import torch
from zgraph.core import FactorNode, SignalNode, SignalNodes, ConstantNode, BaseLeafNode, DynamicLeafNode
from zgraph.transforms import graph_to_function, legendre_transform
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
T, mu1, mu2 = SignalNodes(0,1,2)

In [4]:
R = 8.314
RT = FactorNode([[R]], [T])
mu1A = FactorNode([2, -1], [RT, mu1] )
mu2A = FactorNode([-1], [mu2])
mu1B = FactorNode([-1], [mu1])
mu2B = FactorNode([1, -1], [RT, mu2] )

In [5]:
phaseA = FactorNode(torch.eye(2), [mu1A, mu2A], beta=RT)
phaseB = FactorNode(torch.eye(2), [mu1B, mu2B], beta=RT)
system = FactorNode(torch.eye(2), [phaseA, phaseB], beta=0)

# Thermodynamic plots (Grand potential & Free energy)

In [6]:
T_val = torch.tensor(298.15)
mu1 = torch.linspace(-10*R*300, 10*R*300, steps=500)
mu2 = -mu1
T_flat = T_val.expand_as(mu1)
input_tensor = torch.stack([T_flat, mu1, mu2], dim=-1)

In [7]:
phaseAb, phaseBb, systemb = graph_to_function([phaseA, phaseB, system], compile=True)

fcns = legendre_transform([phaseA, phaseB, system], [1, 2])
fA, fB, fsys = graph_to_function(fcns, compile=True)



In [8]:
gA_vals = phaseAb(input_tensor).detach().cpu().numpy().squeeze()
gB_vals = phaseBb(input_tensor).detach().cpu().numpy().squeeze()
gsys_vals = systemb(input_tensor).detach().cpu().numpy().squeeze()

In [9]:
fAd, muAd = fA(input_tensor)
fBd, muBd = fB(input_tensor)
fsysd, musysd = fsys(input_tensor)

In [10]:
# Evaluate data

x_mu = input_tensor[..., 1].detach().cpu().numpy().squeeze()

xA_frac = -muAd[:, 1].detach().cpu().numpy().squeeze()
yA_free = -fAd.detach().cpu().numpy().squeeze()

xB_frac = -muBd[:, 1].detach().cpu().numpy().squeeze()
yB_free = -fBd.detach().cpu().numpy().squeeze()

xsys_frac = -musysd[:, 1].detach().cpu().numpy().squeeze()
ysys_free = -fsysd.detach().cpu().numpy().squeeze()

# Create stacked subplots (2 rows, 1 col) with linked legend
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=("Grand potential plot", "Free energy plot")
)

colors = {
    "phase A": "#1f77b4",
    "phase B": "#ff7f0e",
    "Equilibrium": "#2ca02c"
}

# --- Top Plot: Grand potential ---
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gA_vals,
        name="phase A",
        legendgroup="phase A",
        showlegend=True,
        line=dict(width=3, color=colors["phase A"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gB_vals,
        name="phase B",
        legendgroup="phase B",
        showlegend=True,
        line=dict(width=3, color=colors["phase B"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gsys_vals,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=True,
        line=dict(width=3, color=colors["Equilibrium"])
    ),
    row=1, col=1
)

# --- Bottom Plot: Free energy ---
fig.add_trace(
    go.Scatter(
        x=xA_frac, y=yA_free,
        name="phase A",
        legendgroup="phase A",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase A"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xB_frac, y=yB_free,
        name="phase B",
        legendgroup="phase B",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase B"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xsys_frac, y=ysys_free,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["Equilibrium"])
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Chemical potential difference, Δμ", row=1, col=1)
fig.update_yaxes(title_text="Grand potential, Ω", row=1, col=1)

fig.update_xaxes(title_text="Mole fraction", row=2, col=1)
fig.update_yaxes(title_text="Free energy", row=2, col=1)

fig.update_layout(
    height=750,
    width=800,
    template="plotly_white"
)

fig.show()